# Metric Boxplots

Aggregates per-spot metrics (cosine, PCC, RMSE, JSD, SSIM) from `../deconv_results/**/` and plots boxplots. Each CSV should be named `*_spot_<metric>.csv` with the corresponding value column.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

base_dir = Path.cwd().parent / "deconv_results"
print(f"Looking for metric CSVs under: {base_dir}")

metrics = {
    "cosine": {"pattern": "*_spot_cosine_similarity.csv", "column": "cosine_similarity"},
    "pcc": {"pattern": "*_spot_pcc.csv", "column": "pcc"},
    "rmse": {"pattern": "*_spot_rmse.csv", "column": "rmse"},
    "jsd": {"pattern": "*_spot_jsd.csv", "column": "jsd"},
    "ssim": {"pattern": "*_spot_ssim.csv", "column": "ssim"},
}

records = []
for metric, cfg in metrics.items():
    suffix = cfg["pattern"].replace("*", "")
    for csv_path in base_dir.glob(f"**/{cfg['pattern']}"):
        df = pd.read_csv(csv_path)
        if cfg["column"] not in df.columns:
            print(f"[skip] {csv_path} missing column {cfg['column']}")
            continue
        sample = csv_path.name.replace(suffix, "")
        for val in df[cfg["column"]].values:
            records.append({"sample": sample, "metric": metric, "value": val})

metrics_df = pd.DataFrame(records)
print(metrics_df.groupby('metric')['value'].describe())
metrics_df.head()

In [ ]:
if metrics_df.empty:
    raise ValueError("No metric files found. Check deconv_results path and filenames.")

sns.set(style="whitegrid")
fig, axes = plt.subplots(1, len(metrics), figsize=(4 * len(metrics), 5), sharey=False)
if len(metrics) == 1:
    axes = [axes]

for ax, (metric, cfg) in zip(axes, metrics.items()):
    sub = metrics_df[metrics_df["metric"] == metric]
    sns.boxplot(data=sub, y="value", ax=ax, color="#4c72b0")
    sns.stripplot(data=sub, y="value", ax=ax, color="black", alpha=0.35, size=2)
    ax.set_title(metric.upper())
    ax.set_xlabel("")
    ax.set_ylabel("Value")

plt.tight_layout()
plt.show()